# 31. 窗口计算与探索性分析

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 10 / 10 步：沿时间观察变化与趋势**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 数据合并与结构转换  →  **本章任务：** 窗口计算与探索性分析  →  **下一步：** 模块大作业《电商履约异常追踪台》
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

一列长长的销售记录看不出趋势，可能只记得总数。



## 本章目标

学完本章，你将能够：

- **理解**：理解窗口/滚动计算与探索性分析思路。
- **操作**：能用 rolling/窗口函数做滑动统计与趋势分析。
- **迁移**：能从时序明细中算出滑动均值等指标并发现经营趋势。


## 31.1 核心概念

**背景引入**：一列长长的销售记录看不出趋势，可能只记得总数。窗口计算给每个时间点配上“近几天均值”，趋势和波动一眼就能读出来；描述性统计、频数、相关与分位数则帮你掂量这批数据“高不高、乱不乱、和谁一起变”。学会这些，你就能又快又准地回答“最近卖得怎么样”这样的问题。

- 窗口大小必须与业务周期一致。
- 滚动统计前要按时间排序。
- 相关系数只描述线性共同变化，不能证明因果。

> **直观类比**：滚动窗口就像在时间轴上放一个“固定宽度的相框”，每往后挪一格、只装框里最近几天的值算平均——开头几天框不满时会先留空（min_periods 可放宽）；shift 则是把整列“整体挪一格”，拿上一天跟今天逐行相减，看清增量。


## 31.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| rolling() | `pd.Series()`、`sales.rolling()`、`.mean()` | rolling用于计算连续窗口统计，窗口大小要符合业务周期。 | 未按日期排序就计算滚动窗口 |
| cumsum() | `pd.Series()`、`sales.cumsum()` | 累计指标保留每个时间点的进展，不只是最终总和。 | 窗口长度与业务周期不一致 |
| pct_change() | `pd.Series()`、`sales.pct_change()`、`.round()` | 增长率要说明比较周期，并检查第一行缺失值。 | 把相关系数解释为因果效应 |
| describe() | `pd.Series()`、`values.describe()` | describe快速查看计数、均值、分位数和最大最小值。 | 未按日期排序就计算滚动窗口 |
| value_counts() | `pd.Series()`、`regions.value_counts()`、`.round()` | 频数统计适合检查分类分布，也可以计算占比。 | 窗口长度与业务周期不一致 |
| corr() 与 rank() | `pd.DataFrame()`、`data.corr()`、`.rank()`、`data['rank']` | 相关系数描述共同变化，rank用于生成排序名次，二者都不能直接证明因果。 | 把相关系数解释为因果效应 |


## 31.3 示例 1：滚动与累计计算

min_periods控制窗口不足时是否返回结果。

**背景引入**：单看每天的销售额，忽高忽低看不出走势。要回答“最近 3 天平均卖多少”“到今天累计卖了多少”“比昨天涨了多少”，就得动三种各有分工的计算：滚动、累计、环比。

**讲解**：rolling 开一个固定长度窗口滑着算 mean，cumsum 逐日累加，pct_change 算相邻两天的变化率。

- 算滚动窗口前务必按日期排好序，否则窗口里混进乱序数据，结果全错；
- `rolling(3, min_periods=1)` 表示窗口 3 天，数据不足 3 天时只要有 1 个值就算，开头两行也有结果；
- `pct_change()` 第一行没有前一天，结果是 NaN，汇报增长率时要说明比较周期；
- **口诀**：rolling 看近期、cumsum 看进展、pct_change 看涨跌，先排序再开窗。


<!-- math-foundation:chapter-31 -->
### 数学推导｜滚动窗口把局部历史变成基准

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜定义时点 $t$ 的历史窗口。** 长度为 $k$ 的窗口是

$$
W_t=\{x_{t-k+1},\ldots,x_t\}
$$

**第 2 步｜在窗口内求局部均值。** 窗口和 $S_t^{(k)}=\sum_{j=0}^{k-1}x_{t-j}$，除以有效观察数就得到移动平均。

**第 3 步｜避免把当期答案放进基准。** 若要用“此前 $k$ 期”预测或判断当期，应计算

$$
Baseline_t=MA_{t-1}^{(k)}
$$

代码上就是先 `rolling(k).mean()` 再 `shift(1)`。

**把上面的关系收束为本章计算式：**

$$
MA_t^{(k)}=\frac{1}{k}\sum_{j=0}^{k-1}x_{t-j}
$$

**符号解释：** $k$ 是窗口长度，$MA_t^{(k)}$ 是时点 $t$ 的 $k$ 期移动平均。

**代码对应：** 先按时间排序，再用 `series.rolling(k).mean()`；需要历史基准时再配合 `shift(1)`。

**使用边界：** 窗口开头样本不足，且滚动统计不能使用未来数据；必须明确 `min_periods`。


In [ ]:
import pandas as pd

sales = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=10, freq="D"),
        "amount": [120, 135, 128, 160, 175, 168, 190, 205, 198, 220],
    }
)
sales["rolling_3d"] = sales["amount"].rolling(3, min_periods=1).mean()
sales["cumulative"] = sales["amount"].cumsum()
sales["daily_growth"] = sales["amount"].pct_change()
print(sales.round(3))


## 31.4 示例 2：描述性统计与频数

数值概览和类别频数应一起检查。

**背景引入**：拿到一沓销售数据，第一反应就是“体检”：金额普遍多大、单子几件、华东客户占几成。光看前几行看不出分布，describe 一行给数值概览，value_counts 摸清分类分布，两个一起看才算摸清底细。

**讲解**：describe 对数值列输出计数、均值、标准差和分位数；value_counts 统计分类频数，normalize=True 输出占比。

- `orders[["amount","items"]].describe()` 只挑数值列做概览，`.round(2)` 控制小数位；
- `value_counts()` 数的是每个地区出现多少次，`normalize=True` 变成占比更直观；
- 数值概览和类别频数要一起看：均值会被极端值拉高，结合分位数判断更稳；
- **口诀**：数值看 describe，分类看 value_counts，normalize 出占比，结合看才全面。


In [ ]:
orders = pd.DataFrame(
    {
        "region": ["华东", "华南", "华东", "华北", "华东", "华南"],
        "amount": [320, 880, 460, 1250, 720, 540],
        "items": [2, 4, 1, 5, 3, 2],
    }
)
print(orders[["amount", "items"]].describe().round(2))
print(orders["region"].value_counts(normalize=True).round(3))


## 31.5 示例 3：相关与分位数

先检查散点和异常，再解释相关系数。

**背景引入**：想知道“金额高的订单，件数是不是也多”，或者给金额排个名次看谁是大单。相关系数描述两个指标是否一起变化，分位数展示金额分布，rank 生成名次——但要记住：相关不等于因果。

**讲解**：quantile 按分位切分布，corr 算两列相关系数，rank 按指定方向生成名次。

- `quantile([0.25, 0.5, 0.75])` 给出下四分位、中位数、上四分位，一眼看分布；
- `corr()` 取值在 -1 到 1 之间，越接近 1 表示一起变大、越接近 -1 表示背道而驰；
- `rank(ascending=False, method="dense")` 金额越大名次越靠前，并列名次不跳号；
- 先看散点和异常值再下结论，相关系数只描述共同变化，不能证明因果关系；
- **口诀**：分位看分布、corr 看共变、rank 排名次，相关不等于因果。


In [ ]:
print("分位数:\n", orders["amount"].quantile([0.25, 0.5, 0.75]))
print("相关矩阵:\n", orders[["amount", "items"]].corr().round(3))
orders["amount_rank"] = orders["amount"].rank(ascending=False, method="dense")
print(orders.sort_values("amount_rank"))


## 31.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# rolling()
# rolling用于计算连续窗口统计，窗口大小要符合业务周期。
import pandas as pd

sales = pd.Series([120, 135, 128, 160])
print(sales.rolling(3, min_periods=1).mean())


In [ ]:
# cumsum()
# 累计指标保留每个时间点的进展，不只是最终总和。
import pandas as pd

sales = pd.Series([120, 150, 180])
print(sales.cumsum())


In [ ]:
# pct_change()
# 增长率要说明比较周期，并检查第一行缺失值。
import pandas as pd

sales = pd.Series([100, 120, 108])
print(sales.pct_change().round(3))


In [ ]:
# describe()
# describe快速查看计数、均值、分位数和最大最小值。
import pandas as pd

values = pd.Series([10, 12, 13, 15, 20])
print(values.describe())


In [ ]:
# value_counts()
# 频数统计适合检查分类分布，也可以计算占比。
import pandas as pd

regions = pd.Series(["华东", "华南", "华东", "华北"])
print(regions.value_counts())
print(regions.value_counts(normalize=True).round(2))


In [ ]:
# corr() 与 rank()
# 相关系数描述共同变化，rank用于生成排序名次，二者都不能直接证明因果。
import pandas as pd

data = pd.DataFrame({"amount": [320, 880, 460], "items": [2, 4, 1]})
print(data.corr())
data["rank"] = data["amount"].rank(ascending=False)
print(data)


**练一练 27.6**：用 `sales = pd.Series([10, 20, 15, 25, 30])` 表示一周五天的销售额，完成三件事：① 计算窗口为 3、`min_periods=1` 的滚动平均，保存为 `rolling_avg`；② 计算累计销售额，保存为 `cum_total`；③ 用 `regions = pd.Series(["华东", "华南", "华东", "华北", "华东"])` 统计地区频数并保存为 `counts`。数据用简单的数值即可。


In [ ]:
# 请在下方填写代码
import pandas as pd

# TODO ①: 计算窗口为 3、min_periods=1 的滚动平均，保存在 rolling_avg
# TODO ②: 计算累计销售额，保存在 cum_total
# TODO ③: 统计 regions 的地区频数，保存在 counts


In [ ]:
import pandas as pd

sales = pd.Series([10, 20, 15, 25, 30])
rolling_avg = sales.rolling(
    3, min_periods=1
).mean()  # ① 3 日滚动平均，前两期窗口不足也返回
cum_total = sales.cumsum()  # ② 累计销售额
regions = pd.Series(["华东", "华南", "华东", "华北", "华东"])
counts = regions.value_counts()  # ③ 地区频数统计
print("滚动平均:\n", rolling_avg)
print("累计销售额:", cum_total.iloc[-1])


## 31.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：",
    f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB",
)
large_orders.head()


In [ ]:
daily_sales = (
    large_orders.query("status == '完成'")
    .set_index("order_time")["sales"]
    .resample("D")
    .sum()
    .to_frame("sales")
)
daily_sales["rolling_7d"] = (
    daily_sales["sales"].rolling(7, min_periods=1).mean()
)
daily_sales["rolling_30d"] = (
    daily_sales["sales"].rolling(30, min_periods=7).mean()
)
daily_sales["growth_7d"] = daily_sales["sales"].pct_change(7)
daily_sales["z_score"] = (
    daily_sales["sales"] - daily_sales["sales"].mean()
) / daily_sales["sales"].std()
print(f"从 {len(large_orders):,} 笔订单得到 {len(daily_sales):,} 天趋势")
display(daily_sales.tail(10).round(3))
display(daily_sales.nlargest(5, "z_score").round(2))


## 31.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 31.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 31.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 31.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 31.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 31.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 31.11 易错点提醒

- 未按日期排序就计算滚动窗口
- 窗口长度与业务周期不一致
- 把相关系数解释为因果效应


## 31.12 练习与作业

1. 创建12个月销售序列
2. 计算3个月移动平均和累计销售
3. 找出销售额最高的3个月

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 31.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建12个月销售序列”。
2. **独立完成**：不复制示例代码，完成“计算3个月移动平均和累计销售”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“找出销售额最高的3个月”，用一两句话说明你修改了什么。

### 31.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 31.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 计算3个月移动平均
# TODO: 计算累计销售
# TODO：请在下方完成 —— 27.13 练习与作业 1. 创建12个月销售序列 2. 计算3个月移动平均和累计销售 3. 找出销售额最高的3个月 提


In [ ]:
import pandas as pd

monthly = pd.DataFrame(
    {
        "month": pd.period_range("2025-01", periods=12, freq="M").astype(
            "string"
        ),
        "sales": [120, 128, 135, 142, 150, 146, 160, 172, 180, 188, 205, 218],
    }
)
monthly["moving_3m"] = monthly["sales"].rolling(3, min_periods=1).mean()
monthly["cumulative"] = monthly["sales"].cumsum()
print(monthly.round(2))
print("Top 3:\n", monthly.nlargest(3, "sales")[["month", "sales"]])


## 31.14 小结

结合滚动窗口、累计指标和描述性统计完成探索性分析。

**迁移思考**：

1. 如果需要计算7日移动平均但前6天数据不足，min_periods 应该设置为多少？
2. 为什么相关系数高不代表因果关系？请举一个相关但无因果的例子。



### 31.14.1 你已经掌握

- 计算滚动与累计指标
- 生成描述性统计
- 检查频数和分位数
- 分析相关关系但避免因果误读



### 31.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 31.14.3 需要注意

- 未按日期排序就计算滚动窗口
- 窗口长度与业务周期不一致
- 把相关系数解释为因果效应



### 31.14.4 完成检查

- [ ] 能够计算滚动与累计指标
- [ ] 能够生成描述性统计
- [ ] 能够检查频数和分位数
- [ ] 能够分析相关关系但避免因果误读



### 31.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

